<a href="https://colab.research.google.com/github/Jasp3r16/00_thesis_generative_timber/blob/main/24_25_optimizer_workflow_with_cost_and_ILP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Thesis: Reclaimed Timber in Deep Generative Design
**Notebook ID:** 24_25_optimizer_workflow_with_cost_and_ILP  
**Author:** Jasper Cluistra   
**Last Updated:** 2026-02-27
### Properties script
**Goal:** to generate a cost matrix for the geometry with use of the timber datasets, then using ILP to find the best matches   
**Inputs:**
*   CSV timber dataset
*   Digital geometry

**Outputs:**
*   Best match for each element in a structure

## Mounting Google drive

In [7]:
import os
import sys
from google.colab import drive

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Definieer je paden (Pas de naam 'Thesis_Project' aan naar jouw mapnaam)
BASE_PATH = '/content/drive/MyDrive/Thesis_TU/'
DATA_PATH = os.path.join(BASE_PATH, 'data_thesis/')
DATA_IMPORT_PATH = os.path.join(BASE_PATH, 'data_import/')
SCRIPT_PATH = os.path.join(BASE_PATH, 'notebooks_thesis/')
OUTPUT_PATH = os.path.join(BASE_PATH, 'research_exports/')

# 3. Voeg de Scripts map toe aan het systeem-pad
# Hierdoor kun je .py bestanden uit die map 'importen'
if SCRIPT_PATH not in sys.path:
    sys.path.append(SCRIPT_PATH)

print(f"✅ Drive mounted. Project directory: {BASE_PATH}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Drive mounted. Project directory: /content/drive/MyDrive/Thesis_TU/


# IMPORTING

## Dataset

In [ ]:
import pandas as pd

# Laad de geëxporteerde dataset in je nieuwe omgeving
# Voeg sep=';' toe aan je inlaad-functie!
df_input_csv = pd.read_csv(DATA_PATH + 'complete_timber.csv', sep=';')

# Print de kolommen om zeker te weten dat ze goed gesplitst zijn
print("Gevonden kolommen:", df_input_csv.columns.tolist())

# Controleer of alle data, inclusief de RS_0000 ID's en binaire states, goed is overgekomen
print("\nDataset succesvol ingeladen. Hier zijn de eerste elementen:\n")
print(df_input_csv.head())

Gevonden kolommen: ['Member_ID', 'State', 'Length_Actual', 'Depth', 'Width', 'E_modulus_eff', 'f_mk', 'Density', 'Embodied Carbon Coëfficiënt', 'Transport_Dist', 'Emmisiefactor', 'Bewerkingsfactor']

Dataset succesvol ingeladen. Hier zijn de eerste elementen:

  Member_ID  State  Length_Actual  Depth  Width  E_modulus_eff  f_mk  Density  \
0  NS_00000      0         2400.0  100.0   38.0        11000.0    24      420   
1  NS_00001      0         2400.0  100.0   50.0        11000.0    24      420   
2  NS_00002      0         2400.0  100.0   75.0        11000.0    24      420   
3  NS_00003      0         2400.0  100.0  100.0        11000.0    24      420   
4  NS_00004      0         2400.0  150.0   38.0        11000.0    24      420   

   Embodied Carbon Coëfficiënt  Transport_Dist  Emmisiefactor  \
0                        150.0             500         0.1473   
1                        150.0             500         0.1019   
2                        150.0             500         0.

## Search space

De search space wordt vanuit het geometrie script geimporteerd, dan wordt een willekeurige samenstelling gekozen als beginpunt van de optimalisatie.


In [13]:
import json

json_path = DATA_IMPORT_PATH + 'search_space.json'
# Lees het "contract" in voor je optimizer
with open(json_path, 'r') as f:
    optimizer_search_space = json.load(f)

print(f"✅ Search Space ingeladen! De optimizer heeft {len(optimizer_search_space)} parameters om aan te draaien.")
# Nu kun je deze variabele direct aan je Optuna, PyTorch of Genetisch Algoritme voeren!

✅ Search Space ingeladen! De optimizer heeft 18 parameters om aan te draaien.


Dit script is bedoeld voor je Colab-omgeving. Het leest de search_space.json in, stelt de parameters dynamisch in op basis van jouw regels, en gebruikt een "dummy" voorspelling (waar later je echte Neurale Netwerk komt) om het beste ontwerp te vinden.

In [15]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 413.9/413.9 kB 7.2 MB/s eta 0:00:00


In [16]:
import json
import optuna
import numpy as np

# ==========================================
# 1. HET CONTRACT INLADEN (De Search Space)
# ==========================================
with open(json_path, 'r') as f:
    ai_search_space = json.load(f)

print(f"✅ Search space ingeladen met {len(ai_search_space)} variabelen.")

# ==========================================
# 2. DE OBJECTIEFFUNCTIE (Wat willen we bereiken?)
# ==========================================
def objective(trial):
    """
    Deze functie wordt door Optuna duizenden keren aangeroepen.
    Elke keer stopt Optuna (de 'trial') er andere parameter-waarden in.
    """

    # A. Bouw de huidige iteratie (de 1D Parameter Vector / X) op basis van de JSON regels
    current_design_X = []

    for var_name, rules in ai_search_space.items():
        if rules['type'] == 'continuous':
            # Optuna mag een random decimaal getal kiezen tussen min en max
            val = trial.suggest_float(var_name, rules['min'], rules['max'])

        elif rules['type'] == 'discrete':
            # Optuna MOET één van de specifieke stappen kiezen uit de lijst
            val = trial.suggest_categorical(var_name, rules['options'])

        current_design_X.append(val)

    # B. Stuur dit ontwerp naar je Surrogate Model (Neuraal Netwerk)
    # Normaal doe je hier: voorspelling = surrogate_model.predict([current_design_X])
    # Omdat we het model nu niet hebben ingeladen, maken we een 'dummy' wiskundige score:

    # --- DUMMY EVALUATIE (Vervang dit later door je AI-voorspelling) ---
    # Stel: we willen dat alle variabelen zo dicht mogelijk bij 0.0 komen,
    # omdat dat (fictief) de minste CO2 oplevert.
    fictieve_co2_score = sum(abs(x) for x in current_design_X)
    # ------------------------------------------------------------------

    # C. Geef de score terug aan Optuna.
    # Optuna zal de volgende keer proberen deze score nog LAGER te krijgen.
    return fictieve_co2_score

# ==========================================
# 3. DE OPTIMALISATIE STARTEN
# ==========================================
print("\n🚀 Starten met Bayesiaanse Optimalisatie...")

# We vertellen Optuna dat we de score willen MINIMALISEREN (bijv. zo min mogelijk CO2/doorbuiging)
# Wil je de 'utilization' juist maximaliseren? Gebruik dan direction='maximize'
study = optuna.create_study(direction='minimize')

# Laat de AI 100 verschillende iteraties proberen (in het echt kun je dit op 5000 zetten)
study.optimize(objective, n_trials=100)

# ==========================================
# 4. RESULTATEN BEKIJKEN
# ==========================================
print("\n🏆 --- OPTIMALISATIE VOLTOOID ---")
print(f"Beste gevonden score (bijv. kg CO2): {study.best_value:.4f}")
print("\nBeste Geometrie (Parameter Vector) om in te laden in Grasshopper:")

# Print de ideale parameters die Optuna heeft gevonden
for var_name, best_val in study.best_params.items():
    print(f"  - {var_name}: \t {best_val:.3f}")

[I 2026-03-05 16:22:54,070] A new study created in memory with name: no-name-8fc82687-a12a-4c79-959c-336a76ebb9ad
[I 2026-03-05 16:22:54,075] Trial 0 finished with value: 9.096108053796062 and parameters: {'v1_shift_x': 0.75, 'v3_shift_y': 1.125, 'v4_shift_x': 0.0, 'v4_shift_y': -1.125, 'v5_shift_y': 0.0, 'v7_shift_x': 0.75, 'v9_u': 0.4517081234350146, 'v9_v': 0.6534409619605636, 'v9_shift_z': -0.375, 'v10_u': 0.47142448549668525, 'v10_v': 0.30283226965697413, 'v10_shift_z': 0.0, 'v11_u': 0.43618904985811224, 'v11_v': 0.5442361558611192, 'v11_shift_z': -0.75, 'v12_u': 0.38312179898972915, 'v12_v': 0.6031552085378631, 'v12_shift_z': 0.375}. Best is trial 0 with value: 9.096108053796062.
[I 2026-03-05 16:22:54,081] Trial 1 finished with value: 12.079925949707999 and parameters: {'v1_shift_x': -0.375, 'v3_shift_y': -0.75, 'v4_shift_x': -1.125, 'v4_shift_y': -0.75, 'v5_shift_y': -0.75, 'v7_shift_x': -1.125, 'v9_u': 0.5906437976759471, 'v9_v': 0.5127222467424536, 'v9_shift_z': -0.75, 'v10_u

✅ Search space ingeladen met 18 variabelen.

🚀 Starten met Bayesiaanse Optimalisatie...


[I 2026-03-05 16:22:54,279] Trial 15 finished with value: 7.404884562951667 and parameters: {'v1_shift_x': -0.75, 'v3_shift_y': -0.75, 'v4_shift_x': 0.0, 'v4_shift_y': 0.375, 'v5_shift_y': 1.125, 'v7_shift_x': 0.0, 'v9_u': 0.3403322175100565, 'v9_v': 0.5906194257429022, 'v9_shift_z': -0.375, 'v10_u': 0.5237503118576745, 'v10_v': 0.5754377698583791, 'v10_shift_z': 0.0, 'v11_u': 0.32038392272205146, 'v11_v': 0.5040530930809992, 'v11_shift_z': 0.0, 'v12_u': 0.4530045180837199, 'v12_v': 0.7223033040958837, 'v12_shift_z': 0.0}. Best is trial 13 with value: 6.857428187971464.
[I 2026-03-05 16:22:54,305] Trial 16 finished with value: 7.277991542346566 and parameters: {'v1_shift_x': -0.75, 'v3_shift_y': -0.75, 'v4_shift_x': 0.0, 'v4_shift_y': 0.375, 'v5_shift_y': 1.125, 'v7_shift_x': 0.0, 'v9_u': 0.333910075747755, 'v9_v': 0.5572766149302693, 'v9_shift_z': -0.375, 'v10_u': 0.5363683422489363, 'v10_v': 0.5686870587811793, 'v10_shift_z': 0.0, 'v11_u': 0.2574213882432531, 'v11_v': 0.4373218288354


🏆 --- OPTIMALISATIE VOLTOOID ---
Beste gevonden score (bijv. kg CO2): 6.0220

Beste Geometrie (Parameter Vector) om in te laden in Grasshopper:
  - v1_shift_x: 	 0.000
  - v3_shift_y: 	 0.375
  - v4_shift_x: 	 0.375
  - v4_shift_y: 	 0.375
  - v5_shift_y: 	 0.000
  - v7_shift_x: 	 0.000
  - v9_u: 	 0.394
  - v9_v: 	 0.558
  - v9_shift_z: 	 0.750
  - v10_u: 	 0.252
  - v10_v: 	 0.263
  - v10_shift_z: 	 -0.375
  - v11_u: 	 0.308
  - v11_v: 	 0.360
  - v11_shift_z: 	 0.000
  - v12_u: 	 0.281
  - v12_v: 	 0.606
  - v12_shift_z: 	 -0.750


In [21]:
# @title
def generate_edges(num_samples, cells_x, cells_y):
    edges_data = []

    # Bereken hulpparameters
    nodes_x_top = cells_x + 1
    nodes_y_top = cells_y + 1
    num_top_vertices = nodes_x_top * nodes_y_top

    # We itereren door elke sample om de edges per sample vast te leggen
    for sample_id in range(num_samples):

        edge_counter = 0  # Reset edge counter per sample (of wil je unieke ID's over de hele file? Meestal per sample resetten: e0..e127)

        # Hulpfunctie om edge toe te voegen
        def add_edge(u, v):
            nonlocal edge_counter
            edges_data.append({
                "sample_id": sample_id,
                "edge_id": f"e{edge_counter}",
                "V1": u,
                "V2": v,
            })
            edge_counter += 1

        # --- 1. TOP LAYER GRID ---
        # Vertices 0 tot num_top_vertices-1
        for r in range(nodes_y_top):      # loop rijen
            for c in range(nodes_x_top):  # loop kolommen
                current = r * nodes_x_top + c

                # Horizontaal (naar rechts)
                if c < cells_x: # zolang niet de laatste kolom
                    add_edge(current, current + 1)

                # Verticaal (naar beneden, of 'boven' in matrix index)
                if r < cells_y: # zolang niet de laatste rij
                    add_edge(current, current + nodes_x_top)

        # --- 2. BOTTOM LAYER GRID ---
        # Start index is na de laatste top vertex
        start_idx_bottom = num_top_vertices

        # Bottom grid heeft evenveel punten als er cellen zijn (cells_x * cells_y)
        # Maar de grid verbindingen zijn er eentje minder dan het aantal punten
        # Bottom punten zijn een grid van (cells_x) breed bij (cells_y) hoog.

        for r in range(cells_y):
            for c in range(cells_x):
                current = start_idx_bottom + r * cells_x + c

                # Horizontaal (naar rechts)
                if c < cells_x - 1:
                    add_edge(current, current + 1)

                # Verticaal (naar beneden)
                if r < cells_y - 1:
                    add_edge(current, current + cells_x)

        # --- 3. DIAGONALS (Pyramid connections) ---
        # Verbind elke Bottom vertex met de 4 Top vertices erboven
        for r in range(cells_y):
            for c in range(cells_x):
                bottom_node = start_idx_bottom + r * cells_x + c

                # De 4 corresponderende punten in de Top layer
                # Top grid is (cells_x + 1) breed
                top_tl = r * nodes_x_top + c               # Top-Left (of row i)
                top_tr = r * nodes_x_top + (c + 1)         # Top-Right
                top_bl = (r + 1) * nodes_x_top + c         # Bottom-Left (row i+1)
                top_br = (r + 1) * nodes_x_top + (c + 1)   # Bottom-Right

                add_edge(bottom_node, top_tl)
                add_edge(bottom_node, top_tr)
                add_edge(bottom_node, top_bl)
                add_edge(bottom_node, top_br)

    return pd.DataFrame(edges_data)

In [22]:
import pandas as pd

# --- CONFIGURATIE ---
GRID_CELLS_X = 2          # Aantal cellen in X
GRID_CELLS_Y = 2          # Aantal cellen in Y
EDGE_LENGTH = 3.0       # Afmeting van een cel
LAYER_HEIGHT = 1.5      # Afstand tussen top en bottom layer
DIVISIONS = 8             # Aantal stappen voor de discrete verplaatsing
NUM_SAMPLES = 100       # Aantal samples
SCALE_UV = 0.25, 0.75        # Random positie in de cel

GRID = f"{GRID_CELLS_X}x{GRID_CELLS_Y}"

# ==========================================
# 5. GEOMETRISCHE RECONSTRUCTIE VAN HET OPTIMUM
# ==========================================
print("\n📦 Exporteren van het volledige winnende ontwerp naar CSV's...")

# Haal de winnende getallen op (de 'DNA' vector)
best_parameters = study.best_params
OPTIMUM_ID = 9999

def reconstruct_optimum_vertices(params, sample_id):
    """Herbouwt de exacte coördinaten op basis van de Optuna parameters."""
    all_vertices = []
    num_nodes_x_top = GRID_CELLS_X + 1
    num_nodes_y_top = GRID_CELLS_Y + 1

    # Bepaal hoeken voor de support/load logica
    def get_corner_indices(cells_x, cells_y):
        return {
            "bl": 0,
            "br": cells_x,
            "tl": cells_y * (cells_x + 1),
            "tr": cells_y * (cells_x + 1) + cells_x
        }
    corners = get_corner_indices(GRID_CELLS_X, GRID_CELLS_Y).values()

    top_layer_coords = {}
    vertex_idx = 0

    # --- STAP 1: TOP LAYER ---
    for i in range(num_nodes_y_top):
        for j in range(num_nodes_x_top):
            base_x = j * EDGE_LENGTH
            base_y = i * EDGE_LENGTH
            base_z = 0.0

            attribute = "support" if vertex_idx in corners else "load"
            v_name = f"v{vertex_idx}"

            # Haal de specifieke shift op uit de best_parameters.
            # Als hij niet in de dictionary staat (omdat Optuna hem niet als variabele had), is de shift 0.0
            shift_x = params.get(f"{v_name}_shift_x", 0.0)
            shift_y = params.get(f"{v_name}_shift_y", 0.0)

            final_x = base_x + shift_x
            final_y = base_y + shift_y
            final_z = base_z

            top_layer_coords[(i, j)] = {'x': final_x, 'y': final_y, 'z': final_z}

            all_vertices.append({
                "sample_id": sample_id,
                "vertex_index": v_name,
                "layer": "top",
                "attribute": attribute,
                "x": round(final_x, 3),
                "y": round(final_y, 3),
                "z": round(final_z, 3)
            })
            vertex_idx += 1

    # --- STAP 2: BOTTOM LAYER ---
    # def bilinear_interpolate(...) # Plak hierboven eventueel je interpolatie functie als hij niet globaal staat
    for i in range(GRID_CELLS_Y):
        for j in range(GRID_CELLS_X):
            v_name = f"v{vertex_idx}"

            p_bl = top_layer_coords[(i, j)]
            p_br = top_layer_coords[(i, j+1)]
            p_tl = top_layer_coords[(i+1, j)]
            p_tr = top_layer_coords[(i+1, j+1)]

            # Haal de u en v op uit de Optuna dictionary
            u = params.get(f"{v_name}_u", 0.5)
            v = params.get(f"{v_name}_v", 0.5)

            # Bilineaire Interpolatie wiskunde
            x_bot = p_bl['x'] * (1 - u) + p_br['x'] * u
            x_top = p_tl['x'] * (1 - u) + p_tr['x'] * u
            lx = x_bot * (1 - v) + x_top * v

            y_bot = p_bl['y'] * (1 - u) + p_br['y'] * u
            y_top = p_tl['y'] * (1 - u) + p_tr['y'] * u
            ly = y_bot * (1 - v) + y_top * v

            # Z positie
            z_shift = params.get(f"{v_name}_shift_z", 0.0)
            final_z = -LAYER_HEIGHT + z_shift

            all_vertices.append({
                "sample_id": sample_id,
                "vertex_index": v_name,
                "layer": "bottom",
                "attribute": "hinges",
                "x": round(lx, 3),
                "y": round(ly, 3),
                "z": round(final_z, 3)
            })
            vertex_idx += 1

    return pd.DataFrame(all_vertices)

# --- 2. Genereer de Vertices ---
df_opt_vertices = reconstruct_optimum_vertices(best_parameters, OPTIMUM_ID)

# --- 3. Genereer de Edges ---
# Je bestaande generate_edges() functie is PERFECT, want die heeft de parameters niet eens nodig!
# De topologie (welk punt zit aan welk punt) is namelijk voor elk ontwerp hetzelfde.
# We roepen hem simpelweg 1x aan voor dit sample.
df_opt_edges = generate_edges(num_samples=1, cells_x=GRID_CELLS_X, cells_y=GRID_CELLS_Y)

# BELANGRIJK: Omdat generate_edges() altijd bij sample_id=0 begint,
# forceren we de ID naar ons OPTIMUM_ID, zodat hij synchroon loopt met de vertices.
df_opt_edges['sample_id'] = OPTIMUM_ID

# --- 4. Exporteren ---
pad_vertices = 'optimum_vertices.csv'
pad_edges = 'optimum_edges.csv'

df_opt_vertices.to_csv(pad_vertices, index=False)
df_opt_edges.to_csv(pad_edges, index=False)

print(f"✅ Succes! Volledig winnende geometrie ({len(df_opt_vertices)} punten, {len(df_opt_edges)} lijnen) opgeslagen.")
print(f"   -> {pad_vertices}")
print(f"   -> {pad_edges}")


📦 Exporteren van het volledige winnende ontwerp naar CSV's...
✅ Succes! Volledig winnende geometrie (13 punten, 32 lijnen) opgeslagen.
   -> optimum_vertices.csv
   -> optimum_edges.csv


# COST MATRIX

Omdat de pseudo-LCA berekening (uit de vorige stap) de impact van de gehele balk al berekent, heb je nu een methodologische keuze:

Optie A (De dubbele boete): Je telt de LCA-score én de penalty's bij elkaar op. Hiermee straf je afval extra zwaar af ten opzichte van het puur inbrengen van materiaal. Dit dwingt het algoritme agressief naar optimaal materiaalgebruik.

Optie B (De uitsplitsing): Je berekent de LCA-score alleen over het nuttige deel, en gebruikt je nieuwe formules om het verlies in kaart te brengen.

In [ ]:
# ==========================================
# CEL 1: INPUTS EN ALGEMENE PARAMETERS
# ==========================================
import pandas as pd
import numpy as np

# GWP Waarden (kg CO2 eq / kg hout)
GWP_VIRGIN = 0.50
GWP_RECLAIMED = 0.08

# LCA Parameters voor E_cost
PROCESSING_PENALTY_CO2 = 5.0  # kg CO2 boete voor bewerkingen (bijv. ontspijkeren)

print("✅ Parameters succesvol geladen.")

✅ Parameters succesvol geladen.


In [ ]:
# ==========================================
# CEL 2: DE REKENFUNCTIES (MODULES)
# ==========================================

def calculate_pseudo_lca_stock(df_stock):
    """
    Berekent de pseudo-LCA score (E_cost) voor een reeds ingeladen DataFrame
    met de timber stock.
    """
    # Maak een kopie om waarschuwingen (SettingWithCopyWarning) te voorkomen
    # en de originele input dataset zuiver te houden.
    df_lca = df_stock.copy()

    print("Start in-memory pseudo-LCA berekeningen...")

    # Stap A: Bereken Volume in m3 (dimensies zijn in mm, dus delen door 1000)
    df_lca['Volume_m3'] = (df_lca['Length_Actual'] / 1000) * (df_lca['Width'] / 1000) * (df_lca['Depth'] / 1000)

    # Stap B: Material Impact (A1-A3)
    df_lca['Impact_Material_kgCO2'] = df_lca['Volume_m3'] * df_lca['Embodied Carbon Coëfficiënt']

    # Stap C: Transport Impact (A4)
    # Dichtheid (Density) delen door 1000 om van kg/m3 naar ton/m3 te gaan
    df_lca['Impact_Transport_kgCO2'] = df_lca['Volume_m3'] * (df_lca['Density'] / 1000) * df_lca['Transport_Dist'] * df_lca['Emmisiefactor']

    # Stap D: Processing Impact (C3 / A3 Re-processing)
    # Binaire bewerkingsfactor (0 of 1) maal de vaste penalty
    df_lca['Impact_Processing_kgCO2'] = df_lca['Bewerkingsfactor'] * PROCESSING_PENALTY_CO2

    # Stap E: Totale E_cost Berekenen
    df_lca['E_cost_Total_kgCO2'] = (
        df_lca['Impact_Material_kgCO2'] +
        df_lca['Impact_Transport_kgCO2'] +
        df_lca['Impact_Processing_kgCO2']
    )

    return df_lca


def calculate_geometric_penalties(slot, stock_item):
    """
    Module 2: Berekent de CO2-penalty's voor zaagverlies en overdimensionering
    voor één specifieke match tussen een Slot (ontwerp) en Stock (voorraad).
    Geeft een tuple terug: (c_waste, c_overdim).
    """
    # 1. Haal afmetingen op en converteer naar meters
    l_slot = slot['Length_Req'] / 1000.0
    w_req = slot['Width_Req'] / 1000.0
    d_req = slot['Depth_Req'] / 1000.0
    a_req = w_req * d_req

    l_stock = stock_item['Length_Actual'] / 1000.0
    w_stock = stock_item['Width'] / 1000.0
    d_stock = stock_item['Depth'] / 1000.0
    a_stock = w_stock * d_stock

    rho = stock_item['Density'] # kg/m3

    # Bepaal GWP afhankelijk van de status (0=Nieuw, 1=Reclaimed)
    gwp_unit = GWP_RECLAIMED if stock_item['State'] == 1 else GWP_VIRGIN

    # 2. Bereken Zaagverlies (Lengteverlies) in kg CO2 eq
    c_waste = (l_stock - l_slot) * a_stock * rho * gwp_unit

    # 3. Bereken Overdimensionering (Dikteverlies) in kg CO2 eq
    c_overdim = (a_stock - a_req) * l_slot * rho * gwp_unit

    # Voorkom negatieve waardes (door afrondingsfouten of exact passende maten)
    c_waste = max(0, c_waste)
    c_overdim = max(0, c_overdim)

    return c_waste, c_overdim

print("✅ Rekenmodules succesvol gedefinieerd.")

✅ Rekenmodules succesvol gedefinieerd.


## Controle

In [ ]:
# 1. Voer de functie uit op je reeds ingeladen DataFrame
df_stock_with_lca = calculate_pseudo_lca_stock(df_input_csv)

# 2. Bekijk de resultaten van de berekening
print("\nPreview van de berekende E_cost per element:")
display(df_stock_with_lca.sample(10))

Start in-memory pseudo-LCA berekeningen...

Preview van de berekende E_cost per element:


,Member_ID,State,Length_Actual,Depth,Width,E_modulus_eff,f_mk,Density,Embodied Carbon Coëfficiënt,Transport_Dist,Emmisiefactor,Bewerkingsfactor,Volume_m3,Impact_Material_kgCO2,Impact_Transport_kgCO2,Impact_Processing_kgCO2,E_cost_Total_kgCO2
83,NS_00083,0,3000.0,300.0,100.0,11000.0,24,420,150.0,500,0.0840,0,0.090000,13.500000,1.587600,0.0,15.087600
88,NS_00088,0,3300.0,150.0,38.0,11000.0,24,420,150.0,500,0.1368,0,0.018810,2.821500,0.540374,0.0,3.361874
81,NS_00081,0,3000.0,300.0,50.0,11000.0,24,420,150.0,500,0.1425,0,0.045000,6.750000,1.346625,0.0,8.096625
177,RS_00010,1,5161.0,212.0,59.0,11000.0,24,420,15.0,88,0.0804,1,0.064554,0.968307,0.191827,5.0,6.160134
155,NS_00155,0,4000.0,200.0,100.0,11000.0,24,420,150.0,500,0.1429,0,0.080000,12.000000,2.400720,0.0,14.400720
141,NS_00141,0,4000.0,100.0,50.0,11000.0,24,420,150.0,500,0.1113,0,0.020000,3.000000,0.467460,0.0,3.467460
90,NS_00090,0,3300.0,150.0,75.0,11000.0,24,420,150.0,500,0.1365,0,0.037125,5.568750,1.064188,0.0,6.632938
148,NS_00148,0,4000.0,175.0,38.0,11000.0,24,420,150.0,500,0.1204,0,0.026600,3.990000,0.672554,0.0,4.662554
140,NS_00140,0,4000.0,100.0,38.0,11000.0,24,420,150.0,500,0.0993,0,0.015200,2.280000,0.316966,0.0,2.596966
169,RS_00002,1,11646.0,590.0,164.0,9000.0,18,380,15.0,32,0.0898,1,1.126867,16.903004,1.230503,5.0,23.133507




---



In [ ]:
# ==========================================
# CEL 3: HOOFDSCRIPT - BOUW DE COST MATRIX (GEFIXED)
# ==========================================

def build_optimization_matrix(df_design, df_stock_raw):
    print("Start generatie van de CO2 Cost Matrix...")

    # 1. Bereken de basis LCA score (Roep de module uit Cel 2 aan)
    df_stock = calculate_pseudo_lca_stock(df_stock_raw)

    n_slots = len(df_design)
    n_stock = len(df_stock)

    cost_matrix = np.full((n_slots, n_stock), np.inf)

    # Bijhouden hoeveel matches er daadwerkelijk lukken
    succesvolle_matches = 0

    for i in range(n_slots):
        slot = df_design.iloc[i]

        # Bepaal de grootste en kleinste doorsnedemaat van het SLOT (voor rotatie)
        slot_max_dim = max(slot['Width_Req'], slot['Depth_Req'])
        slot_min_dim = min(slot['Width_Req'], slot['Depth_Req'])

        for j in range(n_stock):
            stock_item = df_stock.iloc[j]

            # Bepaal de grootste en kleinste doorsnedemaat van de STOCK
            stock_max_dim = max(stock_item['Width'], stock_item['Depth'])
            stock_min_dim = min(stock_item['Width'], stock_item['Depth'])

            # --- HARD CONSTRAINTS (INCLUSIEF ROTATIE) ---
            if (stock_item['Length_Actual'] >= slot['Length_Req'] and
                stock_max_dim >= slot_max_dim and
                stock_min_dim >= slot_min_dim):

                # Het past! Haal de basis E_cost op
                e_cost_base = stock_item['E_cost_Base']

                # Roep Module 2 aan voor penalty's
                c_waste, c_overdim = calculate_geometric_penalties(slot, stock_item)

                # Totale score
                total_match_score = e_cost_base + c_waste + c_overdim
                cost_matrix[i, j] = total_match_score
                succesvolle_matches += 1

    print(f"✅ Matrix gegenereerd! Dimensies: {n_slots} x {n_stock}.")
    print(f"📊 Aantal fysiek geldige combinaties gevonden: {succesvolle_matches}")

    if succesvolle_matches == 0:
        print("\n⚠️ WAARSCHUWING: 0 matches gevonden. Je voorraad bevat geen elementen die groot genoeg zijn voor je ontwerp.")
        print(f"Grootste element in voorraad: Lengte={df_stock['Length_Actual'].max()}mm, Breedte={df_stock['Width'].max()}mm, Diepte={df_stock['Depth'].max()}mm")

    return cost_matrix, df_stock

# --- UITVOEREN VAN DE WORKFLOW ---

# Let op: Ik heb de test-inputs hier iets kleiner gemaakt,
# zodat ze zeker weten passen in de catalogus die we eerder hebben gegenereerd!
design_data = pd.DataFrame({
    'Element_ID': ['Kolom_1', 'Ligger_1', 'Ligger_2'],
    'Length_Req': [2000, 2500, 2500],  # mm (iets korter gemaakt)
    'Width_Req': [50, 75, 75],         # mm (past binnen je max 100mm TUPLE_WIDTH)
    'Depth_Req': [100, 150, 150]       # mm
})

cost_matrix, verrijkte_stock = build_optimization_matrix(design_data, df_input_csv)

# Print de matrix om te controleren of er nu getallen in staan in plaats van 'inf'
print("\nPreview Cost Matrix (eerste 3 rijen, eerste 5 kolommen):")
print(cost_matrix[:3, :5])

Start generatie van de CO2 Cost Matrix...
Start in-memory pseudo-LCA berekeningen...


KeyError: 'E_cost_Base'

# MATCHING ALGORITHM / ILP

## Input

## Script

In [ ]:
import pulp

# 1. DATA DEFINITIE
# De rijen (Stock / X) en kolommen (Constructie / Y)
stock_items = ['X1', 'X2', 'X3', 'X4N', 'X5N']
construction_slots = ['Y1', 'Y2', 'Y3', 'Y4', 'Y5', 'Y6']

# Jouw Cost Matrix (zoals je die in de tabel gaf)
# Let op: de volgorde moet overeenkomen met stock_items
cost_matrix = [
    [2,  6,  8,  3,  1,  4],   # X1
    [4,  5,  3,  6,  3,  2],   # X2
    [1,  7,  4,  9,  6,  5],   # X3
    [10, 10, 2,  2,  10, 10],  # X4 (Nieuw)
    [10, 1,  1,  10, 10, 10]   # X5 (Nieuw)
]

# We maken een dictionary om makkelijk kosten op te zoeken: costs['X1']['Y1'] = 2
costs = {
    stock_items[i]: {construction_slots[j]: cost_matrix[i][j]
                     for j in range(len(construction_slots))}
    for i in range(len(stock_items))
}

# 2. HET MODEL OPZETTEN
# We willen de kosten MINIMALISEREN
prob = pulp.LpProblem("Reclaimed_Timber_Matching", pulp.LpMinimize)

# 3. DECISION VARIABLES
# Dit maakt voor elke combinatie X-Y een variabele die 0 of 1 kan zijn (Binary)
# x['X1']['Y1'] wordt 1 als we matchen, anders 0.
x = pulp.LpVariable.dicts("Match", (stock_items, construction_slots), 0, 1, pulp.LpBinary)

# 4. OBJECTIVE FUNCTION (Doel)
# Som van alle (kosten * keuze)
prob += pulp.lpSum([x[i][j] * costs[i][j] for i in stock_items for j in construction_slots])

# 5. CONSTRAINTS (De Regels)
# Regel A: Elke Y (constructie onderdeel) MOET precies 1 stuk hout krijgen.
for j in construction_slots:
    prob += pulp.lpSum([x[i][j] for i in stock_items]) == 1

# Regel B: Capaciteit van de Stock (X)
# X1, X2, X3 zijn 'reclaimed' en uniek -> Maximaal 1x gebruiken
reclaimed_limit = 1
for i in ['X1', 'X2', 'X3']:
    prob += pulp.lpSum([x[i][j] for j in construction_slots]) <= reclaimed_limit

# X4, X5 zijn 'nieuw' -> Mogen vaker gebruikt worden (bijv. max 6x, want er zijn 6 gaten)
new_limit = 10000
for i in ['X4N', 'X5N']:
    prob += pulp.lpSum([x[i][j] for j in construction_slots]) <= new_limit

# 6. OPLOSSEN
prob.solve()

# 7. RESULTAAT PRINTEN
print(f"Status: {pulp.LpStatus[prob.status]}")
print("-" * 35)

total_cost = 0
print(f"{'Stock':<10} -> {'Element':<10} {'Kosten'}")

for j in construction_slots:
    for i in stock_items:
        if x[i][j].varValue == 1: # Als de beslissing 'JA' is
            print(f"{j:<10} -> {i:<10} {costs[i][j]}")
            total_cost += costs[i][j]

print("-" * 35)
print(f"Totale 'Penalty' Cost: {total_cost}")

Status: Optimal
-----------------------------------
Stock      -> Element    Kosten
Y1         -> X3         1
Y2         -> X5N        1
Y3         -> X5N        1
Y4         -> X4N        2
Y5         -> X1         1
Y6         -> X2         2
-----------------------------------
Totale 'Penalty' Cost: 8


# EXPORT


📦 Exporteren van het winnende ontwerp naar CSV...
✅ Succes! Winnende parameters opgeslagen als 'optimum_design.csv'

Inhoud van de export:
    sample_id  v1_shift_x  v3_shift_y  v4_shift_x  v4_shift_y  v5_shift_y  \
0  optimum_01         0.0       0.375       0.375       0.375         0.0   

   v7_shift_x      v9_u     v9_v  v9_shift_z     v10_u     v10_v  v10_shift_z  \
0         0.0  0.393748  0.55833        0.75  0.252441  0.263204       -0.375   

      v11_u     v11_v  v11_shift_z     v12_u     v12_v  v12_shift_z  
0  0.307796  0.359961          0.0  0.280871  0.605602        -0.75  
